#What are we building?

A minimal RAG system that can:

* Store documents as embeddings

* Retrieve relevant documents for a query

* Send retrieved context to an LLM

* Generate an answer

This will be our baseline to compare all advanced techniques later.

#Libraries Used

**We’ll use:**

* langchain(orchestration)-glue framework for RAG[standardize documents,connect retrievers,vector stores,LLMs]

* faiss-fast vector search engine[stores embeddings,find nearest vectors quickly]

* sentence-transformers (for embeddings)longcha

* A mock LLM response

In [ ]:
# ==========================
# STEP 0: Base RAG Pipeline
# Vanilla Retrieval-Augmented Generation
# ==========================

# 1️⃣ Install required packages
!pip install -U langchain_huggingface langchain-core sentence-transformers faiss-cpu --quiet


In [ ]:
!pip install langchain_community faiss-cpu

In [ ]:
# 2️⃣ Import libraries (updated for current versions)
from langchain_core.documents import Document
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS



In [ ]:
# 3️⃣ Create sample documents
# Documents + metadata

documents = [
    Document(
        page_content="Transformers use self-attention to process sequences in parallel.",
        metadata={"source": "nlp_notes", "year": 2023}
    ),
    Document(
        page_content="The attention mechanism computes weighted sums of value vectors using query and key vectors.",
        metadata={"source": "nlp_notes", "year": 2022}
    ),
    Document(
        page_content="RNNs process sequences sequentially and suffer from vanishing gradients.",
        metadata={"source": "old_notes", "year": 2019}
    ),
    Document(
        page_content="Transformers outperform RNNs due to better long-range dependency handling.",
        metadata={"source": "blog", "year": 2021}
    ),
    Document(
        page_content="FAISS enables fast similarity search over vector embeddings.",
        metadata={"source": "vector_db_notes", "year": 2022}
    )
]


In [ ]:
# 4️⃣ Create embeddings
# Sentence-Transformers embeddings via HuggingFace
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


In [ ]:
# 5️⃣ Create FAISS vectorstore
# This stores document embeddings and allows similarity search

vectorstore = FAISS.from_documents(
    documents=documents,
    embedding=embedding_model
)


In [ ]:
# 6️⃣ Create a basic retriever
# Pure vector similarity, top-k results

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2}
)


In [ ]:
# 7️⃣ Example user query
query = "Explain attention mechanism in transformers"


In [ ]:
# Using the FAISS vectorstore directly
retrieved_docs = vectorstore.similarity_search(query, k=2)

for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n--- Document {i} ---")
    print(doc.page_content)


In [ ]:
# 9️⃣ Mock LLM to generate answer
# Shows how context is combined (no API required)

def mock_llm_answer(query, docs):
    context = "\n".join([doc.page_content for doc in docs])
    return f"""
QUESTION:
{query}

CONTEXT USED:
{context}

ANSWER:
Based on the context, attention in transformers allows the model to focus on relevant parts of the input using query, key, and value vectors.
"""

# Generate mock answer
print(mock_llm_answer(query, retrieved_docs))
